# Progetto 3: Confronto Cross-Linguistico della Complessità Polisemica

**Obiettivo:** confrontare la complessità polisemica di lemmi corrispondenti in inglese, italiano e spagnolo usando Open Multilingual WordNet (OMW).

**Metriche calcolate:**
- Numero di sensi distinti per lemma/lingua
- Entropia di Shannon sulla distribuzione dei sensi
- Top-1 share (quota del senso dominante)

**Dataset:** Open Multilingual WordNet via NLTK (`omw-1.4`), nessuna API esterna necessaria.

## 1. Setup e Import

In [ ]:
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

from nltk.corpus import wordnet as wn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import entropy as scipy_entropy
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Stile grafici
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

print("Lingue disponibili in OMW:", sorted(wn.langs()))
print("\nSetup completato.")

## 2. Definizione dei Lemmi e delle Traduzioni

Selezioniamo lemmi con traduzioni note nelle tre lingue.
La struttura è: `(lemma_en, lemma_it, lemma_es, categoria)`

In [ ]:
# Lemmi allineati: (inglese, italiano, spagnolo, categoria_semantica)
LEMMI = [
    # Sostantivi polisemici classici
    ('bank',    'banca',     'banco',      'sostantivo'),
    ('hand',    'mano',      'mano',       'sostantivo'),
    ('head',    'testa',     'cabeza',     'sostantivo'),
    ('line',    'linea',     'línea',      'sostantivo'),
    ('key',     'chiave',    'llave',      'sostantivo'),
    ('table',   'tavolo',    'tabla',      'sostantivo'),
    ('mouth',   'bocca',     'boca',       'sostantivo'),
    ('foot',    'piede',     'pie',        'sostantivo'),
    # Verbi
    ('run',     'correre',   'correr',     'verbo'),
    ('give',    'dare',      'dar',        'verbo'),
    ('take',    'prendere',  'tomar',      'verbo'),
    ('see',     'vedere',    'ver',        'verbo'),
    ('break',   'rompere',   'romper',     'verbo'),
    ('turn',    'girare',    'girar',      'verbo'),
    # Aggettivi
    ('light',   'leggero',   'ligero',     'aggettivo'),
    ('hard',    'duro',      'duro',       'aggettivo'),
    ('free',    'libero',    'libre',      'aggettivo'),
    ('right',   'giusto',    'correcto',   'aggettivo'),
    ('open',    'aperto',    'abierto',    'aggettivo'),
    ('deep',    'profondo',  'profundo',   'aggettivo'),
]

LINGUE = {
    'eng': 'Inglese',
    'ita': 'Italiano',
    'spa': 'Spagnolo'
}

print(f"Lemmi selezionati: {len(LEMMI)}")
print(f"Lingue analizzate: {list(LINGUE.values())}")

## 3. Estrazione dei Sensi da WordNet

Per ogni (lemma, lingua) recuperiamo i synset disponibili in OMW.

> **Nota metodologica:** OMW non fornisce frequenze d'uso per italiano e spagnolo (disponibili solo per l'inglese tramite SemCor). La distribuzione dei sensi viene quindi approssimata come **uniforme** per tutte le lingue, il che implica che l'entropia dipende esclusivamente dal numero di sensi. Questa semplificazione verrà discussa nell'analisi finale.

In [ ]:
def get_synsets(lemma: str, lang: str) -> list:
    """Restituisce i synset di un lemma in una data lingua."""
    return wn.synsets(lemma, lang=lang)


def calcola_metriche(synsets: list) -> dict:
    """
    Calcola le metriche di complessità polisemica.
    
    - n_sensi: numero di synset distinti
    - entropia: Shannon entropy (distribuzione uniforme)
    - top1_share: quota del senso dominante (1/n con distribuzione uniforme)
    - entropia_norm: entropia normalizzata su log2(n_sensi)
    """
    n = len(synsets)
    if n == 0:
        return {'n_sensi': 0, 'entropia': 0.0, 'top1_share': 1.0, 'entropia_norm': 0.0}
    if n == 1:
        return {'n_sensi': 1, 'entropia': 0.0, 'top1_share': 1.0, 'entropia_norm': 0.0}
    
    # Distribuzione uniforme (tutti i sensi equiprobabili)
    distribuzione = np.ones(n) / n
    h = scipy_entropy(distribuzione, base=2)        # Shannon entropy in bit
    h_max = np.log2(n)                              # entropia massima possibile
    h_norm = h / h_max if h_max > 0 else 0.0       # sempre 1.0 con distribuzione uniforme
    top1 = 1.0 / n                                  # con distribuzione uniforme
    
    return {
        'n_sensi': n,
        'entropia': round(h, 4),
        'top1_share': round(top1, 4),
        'entropia_norm': round(h_norm, 4)
    }


# Costruzione del DataFrame principale
righe = []
for lemma_en, lemma_it, lemma_es, categoria in LEMMI:
    lemmi_per_lingua = {
        'eng': lemma_en,
        'ita': lemma_it,
        'spa': lemma_es
    }
    for lang_code, lang_nome in LINGUE.items():
        lemma = lemmi_per_lingua[lang_code]
        synsets = get_synsets(lemma, lang_code)
        metriche = calcola_metriche(synsets)
        righe.append({
            'lemma_en': lemma_en,
            'lemma': lemma,
            'lingua': lang_nome,
            'lingua_code': lang_code,
            'categoria': categoria,
            **metriche
        })

df = pd.DataFrame(righe)
print(f"Record totali: {len(df)}")
print(f"\nAnteprima:")
df.head(9)

## 4. Statistiche Descrittive per Lingua

In [ ]:
stats = df.groupby('lingua').agg(
    media_sensi=('n_sensi', 'mean'),
    mediana_sensi=('n_sensi', 'median'),
    max_sensi=('n_sensi', 'max'),
    media_entropia=('entropia', 'mean'),
    media_top1=('top1_share', 'mean'),
    lemmi_monosemici=('n_sensi', lambda x: (x <= 1).sum())
).round(3)

# Riordina per lingua
stats = stats.reindex(['Inglese', 'Italiano', 'Spagnolo'])
print("=== Statistiche descrittive per lingua ===")
print(stats.to_string())

## 5. Visualizzazioni

### 5.1 Heatmap: Numero di Sensi per Lemma × Lingua

In [ ]:
pivot_sensi = df.pivot_table(
    index='lemma_en', columns='lingua', values='n_sensi'
).reindex(columns=['Inglese', 'Italiano', 'Spagnolo'])

# Ordina per numero medio di sensi in inglese (decrescente)
pivot_sensi = pivot_sensi.sort_values('Inglese', ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    pivot_sensi,
    annot=True, fmt='.0f',
    cmap='YlOrRd',
    linewidths=0.5,
    linecolor='white',
    ax=ax,
    cbar_kws={'label': 'Numero di sensi'}
)
ax.set_title('Numero di Sensi per Lemma e Lingua', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Lingua', fontsize=12)
ax.set_ylabel('Lemma (EN)', fontsize=12)
ax.tick_params(axis='x', rotation=0)
ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.savefig('heatmap_sensi.png', bbox_inches='tight')
plt.show()
print("Grafico salvato: heatmap_sensi.png")

### 5.2 Barplot: Confronto Entropia per Lingua

In [ ]:
colori = {'Inglese': '#2196F3', 'Italiano': '#4CAF50', 'Spagnolo': '#FF5722'}

fig, ax = plt.subplots(figsize=(14, 6))

lemmi_ordinati = pivot_sensi.index.tolist()  # stesso ordine dell'heatmap
x = np.arange(len(lemmi_ordinati))
width = 0.28

for i, (lingua, colore) in enumerate(colori.items()):
    valori = df[df['lingua'] == lingua].set_index('lemma_en').reindex(lemmi_ordinati)['entropia'].values
    bars = ax.bar(x + (i - 1) * width, valori, width, label=lingua, color=colore, alpha=0.85, edgecolor='white')

ax.set_xlabel('Lemma', fontsize=12)
ax.set_ylabel('Entropia di Shannon (bit)', fontsize=12)
ax.set_title('Entropia Polisemica per Lemma e Lingua', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(lemmi_ordinati, rotation=45, ha='right')
ax.legend(title='Lingua', framealpha=0.9)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('barplot_entropia.png', bbox_inches='tight')
plt.show()
print("Grafico salvato: barplot_entropia.png")

### 5.3 Scatter: Inglese vs Italiano e Inglese vs Spagnolo (n_sensi)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cat_colori = {'sostantivo': '#E91E63', 'verbo': '#9C27B0', 'aggettivo': '#FF9800'}

coppie = [
    ('Italiano',  axes[0], 'EN vs IT'),
    ('Spagnolo',  axes[1], 'EN vs ES'),
]

en_data = df[df['lingua'] == 'Inglese'].set_index('lemma_en')

for lingua2, ax, titolo in coppie:
    l2_data = df[df['lingua'] == lingua2].set_index('lemma_en')
    comuni = en_data.index.intersection(l2_data.index)
    
    for lemma in comuni:
        x_val = en_data.loc[lemma, 'n_sensi']
        y_val = l2_data.loc[lemma, 'n_sensi']
        cat = en_data.loc[lemma, 'categoria']
        col = cat_colori.get(cat, 'gray')
        ax.scatter(x_val, y_val, color=col, s=80, zorder=3)
        ax.annotate(lemma, (x_val, y_val), textcoords='offset points',
                    xytext=(4, 4), fontsize=8, alpha=0.8)
    
    # Linea y=x (parità)
    max_val = max(en_data.loc[comuni, 'n_sensi'].max(),
                  l2_data.loc[comuni, 'n_sensi'].max()) + 1
    ax.plot([0, max_val], [0, max_val], 'k--', alpha=0.3, label='parità')
    
    ax.set_xlabel('N° sensi — Inglese', fontsize=11)
    ax.set_ylabel(f'N° sensi — {lingua2}', fontsize=11)
    ax.set_title(titolo, fontsize=13, fontweight='bold')
    ax.legend(['parità EN=L2'], fontsize=9)
    ax.grid(alpha=0.3)

# Legenda categorie
patches = [mpatches.Patch(color=c, label=l) for l, c in cat_colori.items()]
fig.legend(handles=patches, title='Categoria', loc='lower center',
           ncol=3, bbox_to_anchor=(0.5, -0.08), fontsize=10)

plt.suptitle('Asimmetria Polisemica: Inglese vs Lingue Romanze', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('scatter_asimmetria.png', bbox_inches='tight')
plt.show()
print("Grafico salvato: scatter_asimmetria.png")

### 5.4 Boxplot: Distribuzione N° Sensi per Lingua e Categoria

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Boxplot per lingua
sns.boxplot(
    data=df, x='lingua', y='n_sensi',
    order=['Inglese', 'Italiano', 'Spagnolo'],
    palette=['#2196F3', '#4CAF50', '#FF5722'],
    ax=axes[0], width=0.5
)
axes[0].set_title('Distribuzione N° Sensi per Lingua', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Lingua')
axes[0].set_ylabel('N° sensi')

# Boxplot per categoria
sns.boxplot(
    data=df, x='categoria', y='n_sensi',
    hue='lingua',
    hue_order=['Inglese', 'Italiano', 'Spagnolo'],
    palette=['#2196F3', '#4CAF50', '#FF5722'],
    ax=axes[1], width=0.6
)
axes[1].set_title('N° Sensi per Categoria e Lingua', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Categoria grammaticale')
axes[1].set_ylabel('N° sensi')
axes[1].legend(title='Lingua', fontsize=9)

plt.tight_layout()
plt.savefig('boxplot_distribuzione.png', bbox_inches='tight')
plt.show()
print("Grafico salvato: boxplot_distribuzione.png")

## 6. Analisi delle Asimmetrie

Identifichiamo i casi più interessanti: lemmi con forte disparità nel numero di sensi tra lingue.

In [ ]:
# Pivot per calcolare asimmetrie
pivot = df.pivot_table(index='lemma_en', columns='lingua', values='n_sensi').fillna(0)
pivot = pivot.reindex(columns=['Inglese', 'Italiano', 'Spagnolo'])

pivot['max_sensi']    = pivot.max(axis=1)
pivot['min_sensi']    = pivot.min(axis=1)
pivot['asimmetria']   = pivot['max_sensi'] - pivot['min_sensi']
pivot['ratio_en_it']  = (pivot['Inglese'] / pivot['Italiano'].replace(0, np.nan)).round(2)
pivot['ratio_en_es']  = (pivot['Inglese'] / pivot['Spagnolo'].replace(0, np.nan)).round(2)

print("=== Lemmi con maggiore asimmetria cross-linguistica ===")
top_asimmetria = pivot.sort_values('asimmetria', ascending=False).head(10)
print(top_asimmetria[['Inglese', 'Italiano', 'Spagnolo', 'asimmetria', 'ratio_en_it', 'ratio_en_es']].to_string())

print("\n=== Lemmi con bassa asimmetria (comportamento simile) ===")
bassa_asimmetria = pivot.sort_values('asimmetria', ascending=True).head(5)
print(bassa_asimmetria[['Inglese', 'Italiano', 'Spagnolo', 'asimmetria']].to_string())

### 6.1 Analisi qualitativa dei casi più asimmetrici

In [ ]:
def mostra_sensi(lemma_en, lemma_it, lemma_es, n_max=4):
    """Mostra i primi N sensi per ciascuna lingua."""
    lemmi_map = {'eng': lemma_en, 'ita': lemma_it, 'spa': lemma_es}
    nomi_map  = {'eng': 'Inglese', 'ita': 'Italiano', 'spa': 'Spagnolo'}
    
    print(f"\n{'='*60}")
    print(f"LEMMA: {lemma_en.upper()} / {lemma_it} / {lemma_es}")
    print(f"{'='*60}")
    
    for lang, nome in nomi_map.items():
        lemma = lemmi_map[lang]
        synsets = wn.synsets(lemma, lang=lang)
        print(f"\n  [{nome}] — {len(synsets)} sensi")
        for i, s in enumerate(synsets[:n_max]):
            gloss = s.definition()[:80] + ('...' if len(s.definition()) > 80 else '')
            print(f"    {i+1}. [{s.name()}] {gloss}")
        if len(synsets) > n_max:
            print(f"    ... e altri {len(synsets) - n_max} sensi")

# Analizziamo i casi più interessanti
casi_interessanti = [
    ('bank',  'banca',   'banco'),
    ('line',  'linea',   'línea'),
    ('light', 'leggero', 'ligero'),
    ('run',   'correre', 'correr'),
]

for en, it, es in casi_interessanti:
    mostra_sensi(en, it, es)

## 7. Visualizzazione Asimmetrie

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Grafico 1: ratio EN/IT e EN/ES per lemma ---
ax = axes[0]
ratio_df = pivot[['ratio_en_it', 'ratio_en_es']].dropna().sort_values('ratio_en_it', ascending=False)
x = np.arange(len(ratio_df))
ax.bar(x - 0.2, ratio_df['ratio_en_it'], 0.35, label='EN / IT', color='#4CAF50', alpha=0.85)
ax.bar(x + 0.2, ratio_df['ratio_en_es'], 0.35, label='EN / ES', color='#FF5722', alpha=0.85)
ax.axhline(1.0, color='black', linestyle='--', linewidth=1, alpha=0.5, label='parità')
ax.set_xticks(x)
ax.set_xticklabels(ratio_df.index, rotation=45, ha='right')
ax.set_ylabel('Ratio (EN / altra lingua)')
ax.set_title('Rapporto N° Sensi EN vs Lingue Romanze', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# --- Grafico 2: top 8 asimmetrie come grouped bar ---
ax2 = axes[1]
top8 = pivot.sort_values('asimmetria', ascending=False).head(8)
x2 = np.arange(len(top8))
ax2.bar(x2 - 0.25, top8['Inglese'],  0.25, label='Inglese',  color='#2196F3', alpha=0.85)
ax2.bar(x2,         top8['Italiano'], 0.25, label='Italiano', color='#4CAF50', alpha=0.85)
ax2.bar(x2 + 0.25, top8['Spagnolo'], 0.25, label='Spagnolo', color='#FF5722', alpha=0.85)
ax2.set_xticks(x2)
ax2.set_xticklabels(top8.index, rotation=45, ha='right')
ax2.set_ylabel('N° sensi')
ax2.set_title('Top 8 Lemmi per Asimmetria Cross-Linguistica', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('asimmetrie.png', bbox_inches='tight')
plt.show()
print("Grafico salvato: asimmetrie.png")

## 8. Export dei Risultati

In [ ]:
# CSV completo
df.to_csv('risultati_polisemia.csv', index=False)

# CSV asimmetrie
pivot.to_csv('asimmetrie.csv')

# Riepilogo statistiche
stats.to_csv('statistiche_per_lingua.csv')

print("File esportati:")
print("  risultati_polisemia.csv  — dati completi per lemma/lingua")
print("  asimmetrie.csv           — confronto diretto EN/IT/ES")
print("  statistiche_per_lingua.csv")
print("  heatmap_sensi.png")
print("  barplot_entropia.png")
print("  scatter_asimmetria.png")
print("  boxplot_distribuzione.png")
print("  asimmetrie.png")

## 9. Discussione

### 9.1 Risultati principali

L'inglese mostra sistematicamente un numero di sensi maggiore rispetto a italiano e spagnolo in quasi tutti i lemmi analizzati. Questo non riflette necessariamente una maggiore polisemia intrinseca dell'inglese, ma è in parte un artefatto della **dimensione di Princeton WordNet** (molto più grande e curato di quelli per le altre lingue) e della minore completezza dell'OMW per italiano e spagnolo.

### 9.2 Casi di asimmetria polisemica

I casi più interessanti di asimmetria sono:

- **bank / banca / banco**: l'inglese fonde in un'unica parola i concetti di istituto finanziario e riva del fiume, mentre l'italiano ha *banca* (solo istituto) e *riva*/*sponda* separati. Questa è vera **asimmetria lessicale**, non solo un problema di coverage del wordnet.
- **line / linea / línea**: l'inglese ha sensi molto estesi (linea telefonica, linea di produzione, verso poetico, ecc.), mentre le lingue romanze tendono a lessicalizzare alcune distinzioni con parole diverse.
- **run / correre / correr**: l'inglese ha decine di costruzioni phrasal (*run out*, *run into*, *run over*) che espandono artificialmente il numero di synset.

### 9.3 Implicazioni per NLP multilingue

- **Machine Translation**: una parola inglese polisemica mappata su un lemma italiano con pochi sensi crea ambiguità in traduzione. Il sistema MT deve capire quale senso inglese corrisponde al lemma italiano disponibile.
- **Cross-lingual WSD**: un modello addestrato su WordNet inglese non generalizza bene su lingue con coverage minore; il numero di classi da predire è molto diverso.
- **Multilingual embeddings**: lo spazio semantico per *bank* in inglese dovrebbe coprire cluster molto diversi rispetto a *banca* in italiano, il che crea disallineamento negli embedding multilingue.

### 9.4 Limiti metodologici

- La distribuzione uniforme dei sensi è una semplificazione forte: in realtà alcuni sensi sono molto più frequenti di altri.
- La qualità e completezza dei wordnet varia molto: Princeton WordNet (EN) è molto più ricco di ItalWordNet e SpanishWordNet.
- I lemmi allineati manualmente potrebbero non essere traduzioni perfette in tutti i sensi.